# Graph Databases — First Contact

Instead of tables and rows, data lives as nodes and relationships. An endpoint IS a node. The connection between two endpoints IS a relationship with its own properties. Graph databases make "who depends on whom" questions trivial — questions that would require 5 JOINs in SQL are one Cypher MATCH away. Neo4j Browser at http://localhost:7474 lets you see the graph visually — paste any query there to get an interactive view of the nodes and edges.

## What makes graph databases different

- **Nodes** — entities (Endpoint, Datacenter, Service); each can carry any set of properties.
- **Relationships** — first-class citizens with direction, type, and their own properties; not foreign keys.
- **No joins** — traversal follows edge pointers directly, O(1) per hop regardless of graph size.
- **When to use** — dependency graphs, fraud detection, network topology, recommendation engines, access control hierarchies.

In [1]:
from pathlib import Path
import sys

for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_neo4j_driver
import pandas as pd

driver = get_neo4j_driver()
with driver.session() as s:
    print(s.run("RETURN 'Connected to Neo4j' AS msg").single()['msg'])

with driver.session() as s:
    labels = [r['label'] for r in s.run("CALL db.labels()")]
    print("Node labels:", labels)

with driver.session() as s:
    rel_types = [r['relationshipType'] for r in s.run("CALL db.relationshipTypes()")]
    print("Relationship types:", rel_types)

Connected to Neo4j
Node labels: ['Endpoint']
Relationship types: ['DEPENDS_ON']


In [2]:
# Count nodes and relationships, then inspect a sample node
with driver.session() as s:
    n = s.run("MATCH (n) RETURN COUNT(n) AS count").single()['count']
    r = s.run("MATCH ()-[r]->() RETURN COUNT(r) AS count").single()['count']
    print(f"Nodes:         {n:,}")
    print(f"Relationships: {r:,}")

with driver.session() as s:
    record = s.run("MATCH (e:Endpoint) RETURN e LIMIT 1").single()
    if record:
        print("\nSample Endpoint node properties:")
        for k, v in dict(record['e']).items():
            print(f"  {k}: {v}")

Nodes:         10,000
Relationships: 19,995

Sample Endpoint node properties:
  hostname: srv-00001.citi.internal
  environment: dev
  service_type: web
  endpoint_id: 50fdd579-de84-4899-a774-a3021d2cfe36
  datacenter: NYC1
  status: active


## 5 Cypher queries against the telemetry graph

Cypher reads like English: `MATCH (n:Label {property: value})-[:RELATIONSHIP]->(m) RETURN n, m`  
The pattern inside `MATCH` is a picture of the subgraph you want to find.

In [3]:
# Query 1 — find all endpoints in NYC1 datacenter
# SQL equivalent: SELECT hostname, service_type, status FROM endpoints WHERE datacenter = 'NYC1' LIMIT 10
with driver.session() as s:
    result = s.run("""
        MATCH (e:Endpoint {datacenter: 'NYC1'})
        RETURN e.hostname    AS hostname,
               e.service_type AS service_type,
               e.status       AS status
        LIMIT 10
    """)
    df1 = pd.DataFrame([dict(r) for r in result])

print("Endpoints in NYC1 (showing 10):")
print(df1.to_string(index=False))

Endpoints in NYC1 (showing 10):
               hostname service_type status
srv-00001.citi.internal          web active
srv-00002.citi.internal      monitor active
srv-00006.citi.internal          web active
srv-00008.citi.internal      monitor active
srv-00009.citi.internal          web active
srv-00012.citi.internal      monitor active
srv-00023.citi.internal        cache active
srv-00025.citi.internal      monitor active
srv-00027.citi.internal       worker active
srv-00032.citi.internal           db active


In [4]:
# Query 2 — find all direct dependencies of one endpoint
# SQL equivalent: SELECT dep.* FROM endpoints e JOIN dependencies d ON e.id=d.from_id JOIN endpoints dep ON dep.id=d.to_id WHERE e.id=X
# Graph: one hop, no join syntax needed

with driver.session() as s:
    rec = s.run("""
        MATCH (e:Endpoint)-[:DEPENDS_ON]->(dep:Endpoint)
        RETURN e.hostname AS source, e.endpoint_id AS source_id
        LIMIT 1
    """).single()
    source_id   = rec['source_id']
    source_host = rec['source']
    print(f"Finding dependencies of: {source_host}")

with driver.session() as s:
    result = s.run("""
        MATCH (e:Endpoint {endpoint_id: $eid})-[:DEPENDS_ON]->(dep:Endpoint)
        RETURN dep.hostname    AS depends_on,
               dep.datacenter  AS datacenter,
               dep.service_type AS type
    """, eid=source_id)
    df2 = pd.DataFrame([dict(r) for r in result])

print(f"Direct dependencies ({len(df2)}):")
print(df2.to_string(index=False))

Finding dependencies of: srv-00001.citi.internal
Direct dependencies (2):
             depends_on datacenter   type
srv-04510.citi.internal       NYC2     db
srv-07180.citi.internal       SNG1 worker


In [5]:
# Query 3 — most critical endpoints (most depended upon)
# Which endpoints have the most other endpoints depending on them?
# If these go down, the most services fail. SQL needs GROUP BY + subquery.
with driver.session() as s:
    result = s.run("""
        MATCH (dep:Endpoint)<-[:DEPENDS_ON]-(e:Endpoint)
        RETURN dep.hostname    AS critical_endpoint,
               dep.datacenter  AS datacenter,
               dep.service_type AS type,
               COUNT(e)         AS dependent_count
        ORDER BY dependent_count DESC
        LIMIT 10
    """)
    df3 = pd.DataFrame([dict(r) for r in result])

print("Most critical endpoints (most depended upon):")
print(df3.to_string(index=False))

Most critical endpoints (most depended upon):
      critical_endpoint datacenter    type  dependent_count
srv-01536.citi.internal       LON1  worker               10
srv-01071.citi.internal       SNG1      db                9
srv-05657.citi.internal       NYC2      db                9
srv-01322.citi.internal       SNG1  worker                9
srv-05532.citi.internal       LON1  worker                8
srv-05861.citi.internal       NYC2 monitor                8
srv-09062.citi.internal       SNG1  worker                8
srv-04637.citi.internal       NYC2   cache                8
srv-08319.citi.internal       NYC1      db                8
srv-05411.citi.internal       NYC1  worker                8


In [6]:
# Query 4 — 2-hop transitive dependency chain
# Find what an endpoint depends on AND what those depend on.
# SQL: two recursive CTEs. Cypher: *1..2 on the relationship.
with driver.session() as s:
    result = s.run("""
        MATCH (e:Endpoint {endpoint_id: $eid})
              -[:DEPENDS_ON*1..2]->(dep:Endpoint)
        RETURN DISTINCT dep.hostname   AS transitive_dependency,
                        dep.datacenter AS datacenter
        LIMIT 15
    """, eid=source_id)
    df4 = pd.DataFrame([dict(r) for r in result])

print(f"Transitive dependencies up to 2 hops ({len(df4)} unique):")
print(df4.to_string(index=False))

Transitive dependencies up to 2 hops (5 unique):
  transitive_dependency datacenter
srv-04510.citi.internal       NYC2
srv-07180.citi.internal       SNG1
srv-04172.citi.internal       NYC1
srv-01674.citi.internal       NYC2
srv-00772.citi.internal       NYC2


In [7]:
# Query 5 — detect circular dependencies (A→B and B→A)
# Classic graph problem — trivial in Cypher, painful in SQL.
# The seed data contains 2 intentional cycles.
with driver.session() as s:
    result = s.run("""
        MATCH (a:Endpoint)-[:DEPENDS_ON]->(b:Endpoint)-[:DEPENDS_ON]->(a)
        RETURN a.hostname AS endpoint_a,
               b.hostname AS endpoint_b
        LIMIT 10
    """)
    cycles = [dict(r) for r in result]

if cycles:
    df5 = pd.DataFrame(cycles)
    print(f"Circular dependencies found ({len(df5)}):")
    print(df5.to_string(index=False))
else:
    print("No circular dependencies found — clean dependency graph.")

Circular dependencies found (2):
             endpoint_a              endpoint_b
srv-04201.citi.internal srv-02365.citi.internal
srv-02365.citi.internal srv-04201.citi.internal


## SQL vs Cypher — same question, different approach

| Question | SQL | Cypher |
|---|---|---|
| All NYC1 endpoints | `WHERE datacenter = 'NYC1'` | `MATCH (e {datacenter:'NYC1'})` |
| Direct dependencies | Self-join on endpoint_id | `MATCH (e)-[:DEPENDS_ON]->(dep)` |
| 2-hop dependencies | Two recursive CTEs | `MATCH (e)-[:DEPENDS_ON*1..2]->(dep)` |
| Circular deps | Complex recursive CTE | `MATCH (a)-[:R]->(b)-[:R]->(a)` |
| Most connected node | `COUNT(*) GROUP BY` + subquery | `COUNT(e) ORDER BY DESC` |
| Any-depth path | Unbounded recursion (dangerous) | `MATCH (a)-[:R*]->(b)` |

## Key observations

- **Traversal is O(1) per hop** — Neo4j follows relationship pointers directly, like linked-list next-pointers. No table scan per hop; cost depends on local neighbourhood size, not total graph size.
- **Relationships are first-class** — they have direction, type, and properties. `DEPENDS_ON` could carry `latency_ms`, `protocol`, or `version`. In SQL that would be a junction table with extra columns.
- **Variable-length paths (`*1..2`)** — one keyword, any depth. SQL needs recursive CTEs that grow in complexity with each hop, and have no clean syntax for "up to N hops".
- **Cycles found** — the seed data contains 2 intentional circular dependencies. In an infrastructure graph these are bugs; in a social graph (A follows B follows A) they're normal. Cypher handles both the same way.
- **Citi hook** — endpoint dependency graphs are exactly this use case. "Which services fail if `srv-07263` goes down?" is one Cypher query with `*1..10`. In SQL that is a recursive CTE with a depth limit and a visited-nodes anti-cycle guard. Neo4j Browser at http://localhost:7474 renders the answer as a visual graph you can click through.